# Single-Qubit Basis Misalignment Diagnostic

Measures how far each qubit's dominant single-qubit RDM eigenvector lies from
the computational Z basis (φ_j = 0° means Z-aligned; φ_j = 90° means maximally off-Z).

**New in this revision:**

- Violin + IQR bar + mean ± 95% bootstrap CI (shown as error bar)
- Per-family summary table with full distribution statistics
- Secondary panel: PRZ vs mean misalignment (validates k/PRZ crossover criterion)


In [ ]:
import sys, csv, json, numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path


def find_repo_root(start=None):
    start = (
        __import__("pathlib").Path.cwd()
        if start is None
        else __import__("pathlib").Path(start).resolve()
    )
    for c in (start, *start.parents):
        if (c / "src").is_dir() and (c / "requirements.txt").exists():
            return c
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

THIS_DIR = REPO_ROOT / "tests"
FIGURE_DIR = THIS_DIR / "figures"
DATA_DIR = THIS_DIR / "results"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

from src.experimentation.runner import (
    exact_statevector,
    make_brickwork,
    make_brickwork_2d,
    make_rfim,
    make_tfim,
    trial_stats,
    bootstrap_ci,
)
from src.visualization.style import mplstyle, polish_axes, save_figure

mplstyle(dpi=180)
mpl.rcParams.update(
    {
        "font.size": 8.0,
        "axes.labelsize": 8.0,
        "xtick.labelsize": 7.0,
        "ytick.labelsize": 7.0,
        "legend.fontsize": 6.8,
        "lines.linewidth": 1.25,
        "lines.markersize": 3.5,
        "axes.linewidth": 0.8,
    }
)

In [ ]:
N = 12
N_TRIALS = 100
SEED_BASE = 20260520
FORCE_RERUN = False

FAMILIES = [
    {
        "key": "tfim",
        "label": "Ordered TFIM",
        "kind": "deterministic",
        "color": "#333333",
        "factory": lambda rng: make_tfim(N, rng, layers=5, J=1.0, h=0.5, dt=0.1),
    },
    {
        "key": "rfim",
        "label": "RFIM (W=2)",
        "kind": "stochastic",
        "color": "#1C875C",
        "factory": lambda rng: make_rfim(N, rng, layers=5, W=2.0, dt=0.2),
    },
    {
        "key": "brickwork_1d",
        "label": "1D Brickwork",
        "kind": "stochastic",
        "color": "#0C5DA5",
        "factory": lambda rng: make_brickwork(N, depth=6, rng=rng),
    },
    {
        "key": "brickwork_2d",
        "label": "2D Brickwork",
        "kind": "stochastic",
        "color": "#8E6DB0",
        "factory": lambda rng: make_brickwork_2d(rows=3, cols=4, depth=4, rng=rng),
    },
]
COLORS = {f["key"]: f["color"] for f in FAMILIES}

In [ ]:
def single_qubit_rdm(sv, n_qubits, qubit):
    psi = np.asarray(sv).reshape([2] * n_qubits)
    mat = np.moveaxis(psi, qubit, 0).reshape(2, -1)
    return mat @ mat.conj().T


def qubit_diagnostics(sv, n_qubits):
    phi = np.empty(n_qubits)
    lam = np.empty(n_qubits)
    for q in range(n_qubits):
        rho = single_qubit_rdm(sv, n_qubits, q)
        ev, evec = np.linalg.eigh(rho)
        dom = evec[:, np.argmax(ev)]
        lmax = float(np.max(ev).real)
        amp0 = min(float(abs(dom[0])), 1.0)
        th0 = 2.0 * np.arccos(amp0)
        phi[q] = min(th0, np.pi - th0)  # distance to nearest Z axis
        lam[q] = lmax
    return phi, lam


def exact_prz(sv):
    p = np.abs(np.asarray(sv)) ** 2
    d = np.sum(p**2)
    return float(1.0 / d) if d > 0 else np.nan


_npz = DATA_DIR / "rotation_diag.npz"
_meta = DATA_DIR / "rotation_diag_meta.json"
_exp = {
    "N": N,
    "n_trials": N_TRIALS,
    "seed_base": SEED_BASE,
    "family_keys": [f["key"] for f in FAMILIES],
}

if _npz.exists() and _meta.exists() and not FORCE_RERUN:
    stored = json.loads(_meta.read_text())
    if stored == _exp:
        raw = dict(np.load(_npz, allow_pickle=False))
        print(f"Loaded cache: {_npz}")
    else:
        raw = None
else:
    raw = None

if raw is None:
    nF = len(FAMILIES)
    phi_all = np.full((nF, N_TRIALS, N), np.nan)
    lam_all = np.full((nF, N_TRIALS, N), np.nan)
    prz_all = np.full((nF, N_TRIALS), np.nan)
    gate_count = np.zeros(nF, dtype=int)

    for fi, fam in enumerate(FAMILIES):
        print(f"\n[{fam['label']}]")
        n_run = 1 if fam["kind"] == "deterministic" else N_TRIALS
        for t in range(n_run):
            seed = SEED_BASE + 10_000 * (fi + 1) + t
            rng = np.random.default_rng(seed)
            circ = fam["factory"](rng)
            gate_count[fi] = len(circ)
            sv = exact_statevector(N, circ)
            phi_t, lam_t = qubit_diagnostics(sv, N)
            phi_all[fi, t] = phi_t
            lam_all[fi, t] = lam_t
            prz_all[fi, t] = exact_prz(sv)
            print(
                f"  t={t:02d} mean_phi={np.degrees(phi_t).mean():.1f}° PRZ={prz_all[fi,t]:.1f}"
            )
        # Duplicate deterministic trial
        if fam["kind"] == "deterministic":
            for t in range(1, N_TRIALS):
                phi_all[fi, t] = phi_all[fi, 0]
                lam_all[fi, t] = lam_all[fi, 0]
                prz_all[fi, t] = prz_all[fi, 0]

    raw = {"phi_rad": phi_all, "lam": lam_all, "prz": prz_all, "gate_count": gate_count}
    np.savez_compressed(_npz, **raw)
    _meta.write_text(json.dumps(_exp, indent=2))
    print(f"Saved {_npz}")

phi_deg = np.degrees(raw["phi_rad"])  # (nF, N_TRIALS, N)
prz_data = raw["prz"]  # (nF, N_TRIALS)

In [ ]:
# ── Rigorous summary table ─────────────────────────────────────────────────────
print("\n" + "=" * 88)
print("Rotation Diagnostics Summary")
print("=" * 88)
hdr = (
    f"{'Family':<18}  {'mean phi':>9}  [95% CI]         "
    f"{'med phi':>8}  IQR               "
    f"{'PRZ mean':>9}  [IQR]"
)
print(hdr)
print("-" * 88)

for fi, fam in enumerate(FAMILIES):
    phi_flat = phi_deg[fi].ravel()  # all trials × all qubits
    prz_flat = prz_data[fi]
    phi_s = trial_stats(phi_flat, n_boot=2000)
    prz_s = trial_stats(prz_flat, n_boot=2000)

    ci_lo = phi_s["med_ci_lo"]
    ci_hi = phi_s["med_ci_hi"]
    print(
        f"{fam['label']:<18}  "
        f"{phi_s['mean']:>8.2f}°  [{ci_lo:.2f}°,{ci_hi:.2f}°]     "
        f"{phi_s['median']:>7.2f}°  [{phi_s['q25']:.2f}°,{phi_s['q75']:.2f}°]     "
        f"{prz_s['median']:>8.1f}  [{prz_s['q25']:.1f},{prz_s['q75']:.1f}]"
    )
print("=" * 88)

# CSV
csv_path = DATA_DIR / "rotation_diag.csv"
with csv_path.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(
        fh,
        fieldnames=[
            "family",
            "mean_phi_deg",
            "med_phi_deg",
            "q25_phi",
            "q75_phi",
            "mean_ci_lo",
            "mean_ci_hi",
            "prz_median",
            "prz_q25",
            "prz_q75",
        ],
    )
    w.writeheader()
    for fi, fam in enumerate(FAMILIES):
        phi_s = trial_stats(phi_deg[fi].ravel(), n_boot=2000)
        prz_s = trial_stats(prz_data[fi], n_boot=2000)
        ci_lo, ci_hi = bootstrap_ci(
            phi_deg[fi].ravel(),
            stat_fn=lambda x: np.mean(x, axis=1) if x.ndim > 1 else np.mean(x),
            n_boot=2000,
        )
        w.writerow(
            {
                "family": fam["label"],
                "mean_phi_deg": phi_s["mean"],
                "med_phi_deg": phi_s["median"],
                "q25_phi": phi_s["q25"],
                "q75_phi": phi_s["q75"],
                "mean_ci_lo": ci_lo,
                "mean_ci_hi": ci_hi,
                "prz_median": prz_s["median"],
                "prz_q25": prz_s["q25"],
                "prz_q75": prz_s["q75"],
            }
        )
print(f"Wrote {csv_path}")

In [ ]:
fig, ax_vio = plt.subplots(figsize=(3.25, 2.6), constrained_layout=True)

positions = np.arange(1, len(FAMILIES) + 1)
angle_data = [phi_deg[fi].ravel() for fi in range(len(FAMILIES))]

# --- Violin ---
parts = ax_vio.violinplot(
    angle_data,
    positions=positions,
    widths=0.72,
    showmeans=False,
    showmedians=False,
    showextrema=False,
)
for body, fam in zip(parts["bodies"], FAMILIES):
    c = fam["color"]
    body.set_facecolor(c)
    body.set_edgecolor(c)
    body.set_alpha(0.18)
    body.set_linewidth(0.8)

for pos, data, fam in zip(positions, angle_data, FAMILIES):
    c = fam["color"]
    q25, med, q75 = np.quantile(data, [0.25, 0.5, 0.75])
    mean = float(np.mean(data))

    # 95% CI for mean (bootstrap)
    ci_lo, ci_hi = bootstrap_ci(
        data,
        stat_fn=lambda x: np.mean(x, axis=1) if x.ndim > 1 else np.array([np.mean(x)]),
        n_boot=2000,
    )

    # IQR bar
    ax_vio.vlines(pos, q25, q75, color=c, linewidth=3.5, alpha=0.92, zorder=4)

    # CI for mean
    ax_vio.vlines(pos, ci_lo, ci_hi, color=c, linewidth=1.0, alpha=0.70, zorder=5)
    ax_vio.scatter(pos, ci_lo, marker="_", color=c, s=40, lw=1.0, zorder=5)
    ax_vio.scatter(pos, ci_hi, marker="_", color=c, s=40, lw=1.0, zorder=5)

    # Median (open circle), mean (diamond)
    ax_vio.scatter(pos, med, color="white", edgecolor=c, s=22, lw=0.9, zorder=6)
    ax_vio.scatter(
        pos, mean, marker="D", color=c, edgecolor="white", s=24, lw=0.5, zorder=7
    )

ax_vio.axhline(0, color="#444", ls=":", lw=0.8, alpha=0.6)
ax_vio.axhline(90, color="#444", ls=":", lw=0.8, alpha=0.6)

ax_vio.text(
    0.02,
    0.05,
    "Z-aligned",
    transform=ax_vio.transAxes,
    fontsize=6.5,
    ha="left",
    va="bottom",
)
ax_vio.text(
    0.02, 0.95, "off-Z", transform=ax_vio.transAxes, fontsize=6.5, ha="left", va="top"
)

ax_vio.set_xticks(positions)
ax_vio.set_xticklabels([f["label"].replace(" ", "\n") for f in FAMILIES], fontsize=6.5)
ax_vio.set_ylabel(r"Misalignment $\phi_j$ (deg.)")
ax_vio.set_ylim(-3, 93)
ax_vio.grid(axis="y", ls=":", alpha=0.30)

summary_handles = [
    Line2D(
        [0],
        [0],
        marker="D",
        ls="none",
        color="#333",
        markerfacecolor="#333",
        markeredgecolor="white",
        ms=4.5,
        label="Mean",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        ls="none",
        color="#333",
        markerfacecolor="white",
        markeredgecolor="#333",
        ms=4.5,
        label="Median",
    ),
    Line2D([0], [0], ls="-", color="#333", lw=3.5, label="IQR"),
    Line2D([0], [0], ls="-", color="#333", lw=1.0, label="95% CI (mean)"),
]

ax_vio.legend(
    handles=summary_handles,
    frameon=False,
    fontsize=6.0,
    loc="upper right",
    labelspacing=0.22,
)

polish_axes(ax_vio)

save_figure(fig, FIGURE_DIR / "fig_rotation_diag")
plt.show()